# 06 · Mobility of remote and onsite workers

Two summaries per device and month, used in the neighbourhood-effects analysis
(`research/04-third-places/neighbourhood-effects/`):

* **radius of gyration** — RMS distance of dwells from the device's centre of mass
  (all dwells, and dwells at POIs only);
* **local entropy** — Shannon entropy of visits to distinct places within
  `mobility.local_radius_m` of home: higher means a more varied local life.

Parameters: `mobility` block of `configs/third_places.json`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from remote_work_utils import third_places as tp
from remote_work_utils.pipeline import output_path
from remote_work_utils.plotting import mean_by
from remote_work_utils.settings import load_settings

settings = load_settings("../configs/demo.json")
config = load_settings("../configs/third_places_demo.json")
months = settings["aggregation"]["months"]
visits = pd.concat([pd.read_parquet(output_path(settings, "visits", m)).assign(month=m) for m in months], ignore_index=True)

In [ ]:
mobility = tp.mobility_from_config(
    visits.rename(columns={"dist_home": "dist_to_home"}),
    config["mobility"],
    home_col="home_census",
    entropy_location_col="geohash",
)
schedule = visits[["identifier", "month", "work_schedule"]].drop_duplicates()
mobility = mobility.merge(schedule, on=["identifier", "month"])
mobility.head()

Metric: radius of gyration (km) and local entropy (nats) · Categories: work schedule ·
X: month. Unit: devices; mean with 95% normal CI.

In [ ]:
mobility["rg_all_km"] = mobility["rg_all"] / 1000
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (column, label) in zip(axes, [("rg_all_km", "Radius of gyration (km)"), ("entropy_local", "Local entropy (nats)")]):
    summary = mean_by(mobility, ["month", "work_schedule"], column)
    for schedule_name, color in [("remote_worker", "#2b6cb0"), ("onsite_worker", "#c53030")]:
        data = summary[summary["work_schedule"] == schedule_name]
        ax.errorbar(data["month"], data["mean"], yerr=[data["mean"] - data["ci_low"], data["ci_high"] - data["mean"]],
                    fmt="o-", capsize=3, color=color, label=schedule_name.replace("_", " "))
    ax.set_xlabel("Month (YYYYMM)")
    ax.set_ylabel(label)
    ax.legend()
axes[0].set_title("Spatial range")
axes[1].set_title("Variety of places near home")
fig.suptitle("Mobility by work schedule")
plt.tight_layout()
plt.show()

> **Fig. 6.1.** Radius of gyration and local entropy by work schedule and month (mean ± 95% CI over
> devices). Remote workers travel over a smaller range. *Notes:* synthetic data; work schedule
> from the remote-work metric (`work_schedule.remote_worker_min_share`); entropy over 7-character
> geohashes within 1 km of home.